# Automatic translation to Tatar using DeepSeek

In [ ]:
# Mount Google Drive (optional — only needed if your files live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import time
import requests

In [ ]:
from google.colab import userdata

DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

In [ ]:
BASE_DIR = "<your-base-dir>"
INPUT_PATH = f"{BASE_DIR}/input/train.conll"
OUTPUT_PATH = f"{BASE_DIR}/output/train_tat.conll"
LOG_PATH = f"{BASE_DIR}/logs/errors.log"

In [ ]:
os.makedirs(f"{BASE_DIR}/input", exist_ok=True)
os.makedirs(f"{BASE_DIR}/output", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)

# create / clear the output file
open(OUTPUT_PATH, "w", encoding="utf-8").close()

In [ ]:
os.makedirs(f"{BASE_DIR}/input", exist_ok=True)
os.makedirs(f"{BASE_DIR}/output", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)

# create / clear the output file
open(OUTPUT_PATH, "w", encoding="utf-8").close()

System prompt

In [ ]:
SYSTEM_PROMPT = """STRICT:

Translation rules:

- Do NOT add comments
- Output ONLY CoNLL format

1. The response ALWAYS starts with:
		# text: <Tatar translation>
		# intent: <intent from the source example>
2. The only forbidden combination is "miña" + "minem" (both meaning "my").
3. Split time expressions into separate entities. For example, 9:15 -> 9 I-datetime, : I-datetime, 15 I-datetime. 8:45ka -> 8 I-alarm/alarm_modifier, : I-alarm/alarm_modifier, 45k-ä I-alarm/alarm_modifier.
4. When translating datetime slots, the choice of form and case must be determined by the semantic role of the time. 4.1. Target time (setting/scheduling: set / add / schedule / alarm / reminder, etc.): use the directional case (-ka / -kä). If am/pm/morning/evening/night is present, do NOT use the adverbs irtän, kichen, tönlä, köndez; instead use the form "morning-/evening-/night-/daytime-adjective + number + -ka / -kä". Example: at 5 am -> irtänge 5-kä, at 7 pm -> kichke 7-gä. 4.2. Descriptive time (moment of an event, a fact, a description): use the locative-temporal case (-ta / -tä); the adverbial forms irtän, kichen, tönlä, köndez are allowed.
5. TOP PRIORITY RULE: when translating, use the most literal possible token-by-token translation from English to Tatar, preserving order and grammatical form (mood, tense, number), unless this violates basic Tatar grammar rules.
6. SECOND MOST IMPORTANT RULE: proper names (people, artists, authors, etc.) MUST be written in Cyrillic (transliteration), with Tatar suffixes where needed. Names of objects (playlists, services, brands, titles of works, etc.) keep their original spelling, i.e. write them in English.
Example 1:
text EN: Add the Matt Murphy tune to the Flow Español playlist.
text TAT: Мэтт Мерфиның көйен Flow Español плейлистына өстә.
Example 2:
text EN: I want to listen to something on Youtube
text TAT: Мин Youtube-та берәр нәрсә тыңларга телим
Example 3:
Matt Murphy -> Мэтт Мерфиның, where "-ның" is a Tatar genitive suffix.
7. Titles of films, series, etc. may also be translated if these works are well-known to the general public in Russia.
8. Series example: Friends -> Друзья
9. The pronoun "my" is translated STRICTLY as the possessive suffix -м, -ем, -ым.
10. In the original English example, "my" is tagged as reference. In the Tatar version, reference tags the word that carries the possessive suffix -м, -ем, -ым. Example: будильнигымны сүндер, where "будильнигымны" is B-reference and "сүндер" is O.
11. Slot entities must stay contiguous, as in the original version. An entity is a name, e.g. datetime, and slots are each word within that entity, i.e. B-datetime and I-datetime mean that one entity spans two slots. If slots get split apart during translation into Tatar, the sentence must be rephrased so the slot stays whole.
Example:
1	бүген	B-datetime
2	Будильнигымны	O
3	кичке	B-datetime
4	5	I-datetime
5	куй	O
Rephrased:
1	Будильнигымны	O
2	бүген	B-datetime
3	кичке	I-datetime
4	5	I-datetime
5	куй	O
12. Replace any words/names with LGBT-related content with neutral words/names.
13. NUMBERS ARE ALWAYS written as digits. Add case endings with a hyphen. NEVER spell out numbers as words (дүрт, биш, ун). Example: 6-дан, 4-тә, 5-кә.
14. Strictly do not add function words if they are not present in the original.
15. If there is a logical connection between entities in the source text (possession, authorship, purpose, source), this connection MUST be expressed explicitly in the Tatar translation (via the genitive case, a possessive suffix, or an equivalent construction).

English example

# text: Add a reminder for today at 4pm
# intent: reminder/set_reminder
1 Add O
2 a O
3 reminder O
4 for O
5 today B-datetime
6 at I-datetime
7 4pm I-datetime

Tatar translation

# text: бүген сәгать дүрткә искәртмә өстә
# intent: reminder/set_reminder
1 бүген B-datetime
2 сәгать I-datetime
3 дүрткә I-datetime
4 искәртмә O
5 өстә O
"""

In [ ]:
def parse_conll(path):
    examples = []
    current = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if line == "":
                if current:
                    examples.append(current)
                    current = []
            else:
                current.append(line)

        if current:
            examples.append(current)

    return examples

In [ ]:
def split_id_and_content(example_lines):
    ex_id = None
    content = []

    for line in example_lines:
        if line.startswith("# id:"):
            ex_id = line
        else:
            content.append(line)

    return ex_id, "\n".join(content)

In [ ]:
def deepseek_chat(system_prompt, user_content, max_tokens=800):
    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"

    payload = {
        "model": "deepseek-chat",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, json=payload, timeout=60)
    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]


In [ ]:
def get_last_processed_id(output_path):
    if not os.path.exists(output_path):
        return None

    last_id = None
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# id:"):
                last_id = line.strip()

    return last_id


In [ ]:
def process_file(resume=True):
    examples = parse_conll(INPUT_PATH)
    total = len(examples)

    last_id = get_last_processed_id(OUTPUT_PATH) if resume else None
    skip = True if last_id else False

    for i, example in enumerate(examples, 1):
        ex_id, content = split_id_and_content(example)

        # skip logic
        if skip:
            if ex_id == last_id:
                skip = False
            continue

        try:
            translated = deepseek_chat(
                SYSTEM_PROMPT,
                content
            )

            with open(OUTPUT_PATH, "a", encoding="utf-8") as out:
                if ex_id:
                    out.write(ex_id + "\n")
                out.write(translated.strip() + "\n\n")

            if i % 50 == 0:
                print(f"✅ {i}/{total} done")

            time.sleep(0.7)

        except Exception as e:
            with open(LOG_PATH, "a", encoding="utf-8") as log:
                log.write(f"{ex_id} error: {e}\n")

On the first run

In [ ]:
process_file(resume=False)

def process_file(resume=True):
    examples = parse_conll(INPUT_PATH)
    total = len(examples)

    last_id = get_last_processed_id(OUTPUT_PATH) if resume else None
    skip = True if last_id else False

    for i, example in enumerate(examples, 1):
        ex_id, content = split_id_and_content(example)

        # skip logic
        if skip:
            if ex_id == last_id:
                skip = False
            continue

        try:
            translated = deepseek_chat(
                SYSTEM_PROMPT,
                content
            )

            with open(OUTPUT_PATH, "a", encoding="utf-8") as out:
                if ex_id:
                    out.write(ex_id + "\n")
                out.write(translated.strip() + "\n\n")

            if i % 50 == 0:
                print(f"✅ {i}/{total} done")

            time.sleep(0.7)

        except Exception as e:
            with open(LOG_PATH, "a", encoding="utf-8") as log:
                log.write(f"{ex_id} error: {e}\n")

In [ ]:
process_file(resume=True)

On the first run

In [ ]:
with open("<your-base-dir>/input/train.conll", "r", encoding="utf-8") as fi:
    list_id = []
    for line in fi:
        if line.startswith("# id:"):
            last_id = line.strip()
            list_id.append(last_id)

with open("<your-base-dir>/output/train_tat.conll", "r", encoding="utf-8") as f:
    list_id_tat = []
    for line in f:
        if line.startswith("# id:"):
            last_id_tat = line.strip()
            list_id_tat.append(last_id_tat)

In [ ]:
On subsequent runs, to avoid overwriting previous translations

In [ ]:
train_path = "<your-base-dir>/input/train.conll"
cleaned_path = "<your-base-dir>/output/train_tat.conll"

# 1⃣ collect IDs from train
list_id = []
with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()
            list_id.append(clean_id)

# 2⃣ collect IDs from cleaned
list_id_tat = []
with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()
            list_id_tat.append(clean_id)

# 3⃣ find missing_ids
missing_ids = set(list_id) - set(list_id_tat)

print("Keeping only:", missing_ids)

# 4⃣ read train in blocks
with open(train_path, "r", encoding="utf-8") as f:
    sentences = f.read().strip().split("\n\n")

# 5⃣ keep ONLY missing_ids
filtered_sentences = []

for sent in sentences:
    for line in sent.split("\n"):
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()

            # NOTE: the condition is now reversed
            if clean_id in missing_ids:
                filtered_sentences.append(sent)
            break

print("Blocks before:", len(sentences))
print("Blocks after:", len(filtered_sentences))

# 6⃣ rewrite the file
with open(train_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(filtered_sentences))

print("Done ✅")

# Checking which sentences were not translated

In [ ]:
with open("<your-base-dir>/input/train.conll", "r", encoding="utf-8") as fi:
    list_id = []
    for line in fi:
        if line.startswith("# id:"):
            last_id = line.strip()
            list_id.append(last_id)

with open("<your-base-dir>/output/train_tat.conll", "r", encoding="utf-8") as f:
    list_id_tat = []
    for line in f:
        if line.startswith("# id:"):
            last_id_tat = line.strip()
            list_id_tat.append(last_id_tat)

In [ ]:
train_path = "<your-base-dir>/input/train.conll"
cleaned_path = "<your-base-dir>/output/train_tat.conll"

# 1⃣ collect IDs from train
train_ids = []
with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            train_ids.append(line.replace("# id:", "").strip())

# 2⃣ collect IDs from cleaned
cleaned_ids = []
with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            cleaned_ids.append(line.replace("# id:", "").strip())

# 3⃣ find missing_ids
missing_ids = set(train_ids) - set(cleaned_ids)

print("Missing IDs found:", len(missing_ids))

# 4⃣ extract only the numbers
numbers = []

for item in missing_ids:
    # item looks like "train_366"
    number = item.split("_")[-1]
    numbers.append(int(number))

# 5⃣ sort
numbers.sort()

print("Numbers:")
print(numbers)

# 6⃣ save to file
output_path = "<your-base-dir>/output/missing_numbers.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for n in numbers:
        f.write(str(n) + "\n")

print("Done ✅")

In [ ]:
train_path = "<your-base-dir>/input/train.conll"
cleaned_path = "<your-base-dir>/output/train_tat.conll"

# 1⃣ collect IDs from train
list_id = []
with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()
            list_id.append(clean_id)

# 2⃣ collect IDs from cleaned
list_id_tat = []
with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()
            list_id_tat.append(clean_id)

# 3⃣ find missing_ids
missing_ids = set(list_id) - set(list_id_tat)

print("Keeping only:", missing_ids)

# 4⃣ read train in blocks
with open(train_path, "r", encoding="utf-8") as f:
    sentences = f.read().strip().split("\n\n")

# 5⃣ keep ONLY missing_ids
filtered_sentences = []

for sent in sentences:
    for line in sent.split("\n"):
        if line.startswith("# id:"):
            clean_id = line.replace("# id:", "").strip()

            # NOTE: the condition is now reversed
            if clean_id in missing_ids:
                filtered_sentences.append(sent)
            break

print("Blocks before:", len(sentences))
print("Blocks after:", len(filtered_sentences))

# 6⃣ rewrite the file
with open(train_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(filtered_sentences))

print("Done ✅")

In [ ]:
Sentence order

In [ ]:
train_path = "<your-base-dir>/output/train_tat.conll"

# 1⃣ read the file in blocks
with open(train_path, "r", encoding="utf-8") as f:
    sentences = f.read().strip().split("\n\n")

# 2⃣ renumber
new_sentences = []

for idx, sent in enumerate(sentences, start=1):
    lines = sent.split("\n")
    new_lines = []

    for line in lines:
        if line.startswith("# id:"):
            new_lines.append(f"# id: train_{idx}")
        else:
            new_lines.append(line)

    new_sentences.append("\n".join(new_lines))

print("Total sentences:", len(new_sentences))

# 3⃣ rewrite the file
with open(train_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(new_sentences))

print("IDs successfully reordered ✅")

In [ ]:
train_path = "<your-base-dir>/input/train.conll"
cleaned_path = "<your-base-dir>/output/train_tat.conll"

# 1⃣ collect IDs from train
train_ids = []
with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            train_ids.append(line.replace("# id:", "").strip())

# 2⃣ collect IDs from cleaned
cleaned_ids = []
with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("# id:"):
            cleaned_ids.append(line.replace("# id:", "").strip())

# 3⃣ find missing_ids
missing_ids = set(train_ids) - set(cleaned_ids)

print("Missing IDs found:", len(missing_ids))

# 4⃣ extract only the numbers
numbers = []

for item in missing_ids:
    # item looks like "train_366"
    number = item.split("_")[-1]
    numbers.append(int(number))

# 5⃣ sort
numbers.sort()

print("Numbers:")
print(numbers)

# 6⃣ save to file
output_path = "<your-base-dir>/output/missing_numbers.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for n in numbers:
        f.write(str(n) + "\n")

print("Done ✅")

In [ ]:
print(repr(cleaned_ids[0]))

In [ ]:
missing_in_cleaned = set(train_ids) - set(cleaned_ids)
extra_in_cleaned = set(cleaned_ids) - set(train_ids)

print("In train but not in cleaned:", len(missing_in_cleaned))
print("In cleaned but not in train:", len(extra_in_cleaned))

In [ ]:
extra_in_cleaned = set(cleaned_ids) - set(train_ids)

print("Extra IDs in cleaned:", len(extra_in_cleaned))
print(extra_in_cleaned)

In [ ]:
with open(train_path, "r", encoding="utf-8") as f:
    sentences = f.read().strip().split("\n\n")

In [ ]:
train_path = "<your-base-dir>/input/train.conll"
cleaned_path = "<your-base-dir>/output/train_tat.conll"

def count_ids(path):
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# id:"):
                count += 1
    return count

print("train IDs:", count_ids(train_path))
print("cleaned IDs:", count_ids(cleaned_path))

In [ ]:
from collections import Counter

counter = Counter(cleaned_ids)

duplicates = [item for item, count in counter.items() if count > 1]

print("Number of duplicates:", len(duplicates))
print("Duplicates:", duplicates)